# 24 (MLA) — Prepare & Split

**ML Analyst perspective.** Frame the problem, engineer features, and create a **leak-free** train/test split. The discipline here — split before any model sees the test set — is what separates a trustworthy model from an overfit one.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. The credit dataset

500 synthetic applicants. `inadimplente` (default) is the classification target; `risco_score` (0-100, higher = riskier) is the regression target. Some `renda` values are NULL to exercise the Imputer.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 500
renda = rng.normal(5000, 2000, n).clip(500, 20000)
idade = rng.integers(22, 70, n)
historico = rng.choice(["bom", "regular", "ruim"], n, p=[0.5, 0.3, 0.2])
divida_ratio = rng.beta(2, 5, n) * 0.6
logit = -2.0 + 1.8 * divida_ratio - 0.0002 * renda + 0.6 * (historico == "ruim") - 0.4 * (historico == "bom")
p_default = 1 / (1 + np.exp(-logit))
inadimplente = (rng.random(n) < p_default).astype(float)
risco_score = (50 + 30 * divida_ratio - 0.002 * renda + 8 * (historico == "ruim") - 5 * (historico == "bom") + rng.normal(0, 6, n)).clip(0, 100)
renda = np.where(rng.random(n) < 0.05, np.nan, renda)
credito = session.createDataFrame(pd.DataFrame({
    "cliente_id": range(1, n + 1),
    "renda": renda,
    "idade": idade,
    "historico": historico,
    "divida_ratio": divida_ratio,
    "risco_score": risco_score,
    "inadimplente": inadimplente,
}))
print("rows:", credito.count())
credito.show(3)

## 2. Profile the target

Check class balance and the continuous target's distribution.

In [ ]:
from irispark.functions import count, avg, min, max

credito.groupBy("inadimplente").count().orderBy("inadimplente").show()
credito.select(min("risco_score").alias("min"), avg("risco_score").alias("avg"), max("risco_score").alias("max")).show()

## 3. Feature preparation

Encode the categorical, impute NULLs, and scale numerics — all SQL-executed.

In [ ]:
from irispark.ml.feature import StringIndexer, Imputer, StandardScaler

idx = StringIndexer(inputCol="historico", outputCol="historico_idx").fit(credito).transform(credito)
imp = Imputer(inputCol="renda", outputCol="renda_imp", strategy="mean").fit(idx).transform(idx)
# Standardize ALL numeric features so the logistic logit stays bounded
# (unscaled features -> huge coefficients -> EXP overflow -> MAXNUMBER).
scaled = StandardScaler(inputCol="renda_imp", outputCol="renda_std", withMean=True, withStd=True).fit(imp).transform(imp)
scaled = StandardScaler(inputCol="idade", outputCol="idade_std", withMean=True, withStd=True).fit(scaled).transform(scaled)
scaled = StandardScaler(inputCol="historico_idx", outputCol="historico_idx_std", withMean=True, withStd=True).fit(scaled).transform(scaled)
scaled = StandardScaler(inputCol="divida_ratio", outputCol="divida_ratio_std", withMean=True, withStd=True).fit(scaled).transform(scaled)
scaled.select("cliente_id", "historico", "historico_idx", "renda", "renda_imp", "renda_std", "inadimplente").show(5)

## 4. Leak-free train/test split

`randomSplit` with a fixed seed. The test set is held out — never used for fitting or tuning.

In [ ]:
train, test = scaled.randomSplit([0.8, 0.2], seed=7)
print("train:", train.count(), "| test:", test.count())
print("no overlap:", train.union(test).distinct().count() == scaled.count())

## 5. Persist the splits

Materialize train/test as tables so later notebooks read the same rows.

In [ ]:
train.write.mode("overwrite").saveAsTable("mla_train")
test.write.mode("overwrite").saveAsTable("mla_test")
print("saved mla_train / mla_test")

## 6. Stratification note

If the target is rare, a random split can under-represent it. `sampleBy` gives a stratified view for inspection.

In [ ]:
strat = credito.stat.sampleBy("inadimplente", {0.0: 0.2, 1.0: 0.2}, sampleByColumns=["inadimplente"])
print("stratified sample rows:", len(strat))
print(strat["inadimplente"].value_counts().to_dict())

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")